# Credit Card Approval Prediction - Exploratory Data Analysis

This notebook analyzes applicant profile data and monthly repayment history. The target label is engineered from `STATUS`; it is not assumed to exist in the raw data.

In [ ]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
os.environ.setdefault('MPLCONFIGDIR', str(PROJECT_ROOT / '.matplotlib_cache'))

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from preprocessing.feature_engineering import create_modeling_dataset
from preprocessing.load_data import load_raw_datasets

sns.set_theme(style='whitegrid', palette='Set2')
plt.rcParams['figure.figsize'] = (11, 6)
plt.rcParams['axes.titlesize'] = 15
plt.rcParams['axes.labelsize'] = 12
plt.rcParams['legend.frameon'] = True

## Phase 2: Dataset Loading And Inspection

In [ ]:
application_df, credit_df = load_raw_datasets()

for name, df in {'Application Record': application_df, 'Credit Record': credit_df}.items():
    print(f'\n{name}')
    print('Shape:', df.shape)
    print('Columns:', df.columns.tolist())
    display(df.head())
    display(df.tail())
    display(df.describe(include='all').transpose())
    print('Missing values:')
    display(df.isna().sum())
    print('Duplicate rows:', df.duplicated().sum())

**Interpretation:** The application file contains applicant attributes, while the credit file contains multiple monthly records per applicant. Because `credit_record.csv` has repeated `ID` values, it must be aggregated before merging.

In [ ]:
modeling_df = create_modeling_dataset(application_df, credit_df)
print('Merged modeling dataset shape:', modeling_df.shape)
display(modeling_df.head())
display(modeling_df['APPROVED'].value_counts(normalize=True).rename('class_ratio'))

## Countplots

In [ ]:
def countplot_with_labels(data, column, title, xlabel, rotation=0):
    """Draw a styled countplot with title, axis labels, grid, and legend when target is present."""
    plt.figure(figsize=(11, 6))
    ax = sns.countplot(data=data, x=column, hue='APPROVED', order=data[column].value_counts().index)
    ax.set_title(title, weight='bold')
    ax.set_xlabel(xlabel)
    ax.set_ylabel('Applicant Count')
    ax.grid(axis='y', alpha=0.35)
    ax.legend(title='Approved', labels=['Rejected', 'Approved'])
    plt.xticks(rotation=rotation, ha='right' if rotation else 'center')
    plt.tight_layout()
    plt.show()

countplot_with_labels(modeling_df, 'CODE_GENDER', 'Gender Distribution By Approval Status', 'Gender')
countplot_with_labels(modeling_df, 'NAME_EDUCATION_TYPE', 'Education Distribution By Approval Status', 'Education', 30)
countplot_with_labels(modeling_df, 'NAME_HOUSING_TYPE', 'Housing Distribution By Approval Status', 'Housing Type', 30)
countplot_with_labels(modeling_df, 'CNT_CHILDREN', 'Children Distribution By Approval Status', 'Number Of Children')

**Interpretation:** Countplots show whether approval outcomes differ across gender, education, housing, and family composition categories. These views help identify class imbalance and category-level risk patterns.

## Histograms And Distribution Plots

In [ ]:
numeric_columns = ['AMT_INCOME_TOTAL', 'AGE_YEARS', 'YEARS_EMPLOYED', 'CNT_FAM_MEMBERS']

for column in numeric_columns:
    plt.figure(figsize=(11, 6))
    ax = sns.histplot(data=modeling_df, x=column, hue='APPROVED', kde=True, bins=35)
    ax.set_title(f'{column} Distribution By Approval Status', weight='bold')
    ax.set_xlabel(column)
    ax.set_ylabel('Frequency')
    ax.grid(axis='y', alpha=0.35)
    plt.tight_layout()
    plt.show()

**Interpretation:** Histograms and KDE curves show skewness, outliers, and whether rejected applicants cluster around specific income, age, employment, or family-size ranges.

## Boxplots

In [ ]:
for column in ['AMT_INCOME_TOTAL', 'AGE_YEARS', 'YEARS_EMPLOYED', 'INCOME_PER_FAMILY_MEMBER']:
    plt.figure(figsize=(10, 6))
    ax = sns.boxplot(data=modeling_df, x='APPROVED', y=column)
    ax.set_title(f'{column} Boxplot By Approval Status', weight='bold')
    ax.set_xlabel('Approved Status')
    ax.set_ylabel(column)
    ax.set_xticklabels(['Rejected', 'Approved'])
    ax.grid(axis='y', alpha=0.35)
    plt.tight_layout()
    plt.show()

**Interpretation:** Boxplots make outliers and median differences visible. Income-related plots are especially important because income often has a long right tail.

## Income Analysis

In [ ]:
income_by_type = modeling_df.groupby(['NAME_INCOME_TYPE', 'APPROVED'])['AMT_INCOME_TOTAL'].median().reset_index()
plt.figure(figsize=(12, 6))
ax = sns.barplot(data=income_by_type, x='NAME_INCOME_TYPE', y='AMT_INCOME_TOTAL', hue='APPROVED')
ax.set_title('Median Income By Income Type And Approval Status', weight='bold')
ax.set_xlabel('Income Type')
ax.set_ylabel('Median Annual Income')
ax.legend(title='Approved', labels=['Rejected', 'Approved'])
ax.grid(axis='y', alpha=0.35)
plt.xticks(rotation=25, ha='right')
plt.tight_layout()
plt.show()

**Interpretation:** Income analysis compares central income levels across employment groups and approval outcomes, which helps identify whether income type changes the risk profile.

## Employment Distribution

In [ ]:
plt.figure(figsize=(11, 6))
ax = sns.histplot(data=modeling_df, x='YEARS_EMPLOYED', hue='APPROVED', bins=40, kde=True)
ax.set_title('Employment Duration Distribution By Approval Status', weight='bold')
ax.set_xlabel('Years Employed')
ax.set_ylabel('Applicant Count')
ax.grid(axis='y', alpha=0.35)
plt.tight_layout()
plt.show()

**Interpretation:** Employment duration can reflect applicant stability. The zero-employment group includes people with the employment sentinel value from the raw data.

## Credit History Analysis

In [ ]:
plt.figure(figsize=(10, 6))
status_order = ['X', 'C', '0', '1', '2', '3', '4', '5']
ax = sns.countplot(data=credit_df, x='STATUS', order=status_order)
ax.set_title('Credit Repayment Status Distribution', weight='bold')
ax.set_xlabel('Credit Status')
ax.set_ylabel('Monthly Record Count')
ax.grid(axis='y', alpha=0.35)
plt.tight_layout()
plt.show()

plt.figure(figsize=(11, 6))
ax = sns.boxplot(data=modeling_df, x='APPROVED', y='LATE_PAYMENT_RATIO')
ax.set_title('Late Payment Ratio By Approval Status', weight='bold')
ax.set_xlabel('Approved Status')
ax.set_ylabel('Late Payment Ratio')
ax.set_xticklabels(['Rejected', 'Approved'])
ax.grid(axis='y', alpha=0.35)
plt.tight_layout()
plt.show()

**Interpretation:** Repayment history confirms that the engineered target separates applicants by overdue behavior. These fields are excluded from production model inputs to avoid target leakage.

## Correlation Heatmap

In [ ]:
corr_columns = [
    'CNT_CHILDREN', 'AMT_INCOME_TOTAL', 'CNT_FAM_MEMBERS', 'AGE_YEARS',
    'YEARS_EMPLOYED', 'IS_EMPLOYED', 'INCOME_PER_FAMILY_MEMBER',
    'CHILDREN_RATIO', 'HAS_CONTACT_METHOD', 'APPROVED'
]
plt.figure(figsize=(12, 8))
ax = sns.heatmap(modeling_df[corr_columns].corr(), annot=True, fmt='.2f', cmap='viridis', linewidths=0.5)
ax.set_title('Correlation Heatmap For Numeric Model Features', weight='bold')
plt.tight_layout()
plt.show()

**Interpretation:** The heatmap highlights numeric relationships and helps detect redundant features. Strong correlations should be considered during modeling and explanation.

## Pairplot

In [ ]:
pairplot_sample = modeling_df[['AMT_INCOME_TOTAL', 'AGE_YEARS', 'YEARS_EMPLOYED', 'INCOME_PER_FAMILY_MEMBER', 'APPROVED']].sample(
    n=min(2500, len(modeling_df)), random_state=42
)
grid = sns.pairplot(pairplot_sample, hue='APPROVED', diag_kind='kde', corner=True)
grid.fig.suptitle('Pairplot Of Key Numeric Features By Approval Status', y=1.02, weight='bold')
plt.show()

**Interpretation:** The pairplot shows pairwise separation between approved and rejected applicants. Sampling is used to keep rendering fast while preserving the visual trend.